# Task 3 (soft MoE) + evaluation on Kaggle
Needs the finished Tasks 1-2 run (`TAG=_v2`). Add that notebook version's **output** as an input: right panel -> Input -> Add Input -> Your Work / Notebook Output. Settings: GPU, Internet ON, secret `GH_TOKEN`. Use **Save & Run All (Commit)**.

In [ ]:
import base64, os, subprocess
BRANCH = 'dev'
REPO = 'AfnanRizwan1/GenAI-A1'
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
except Exception as e:
    TOKEN = None
    print(f'GH_TOKEN secret not available ({type(e).__name__}). Private repo: right panel -> Add-ons -> Secrets -> switch GH_TOKEN on for THIS notebook, then restart.')

def git(*args):
    """Run git; the token is sent as a one-off header, never stored in .git/config or printed."""
    cmd, secrets = ['git'], []
    if TOKEN:
        b64 = base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode()
        cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {b64}']
        secrets = [TOKEN, b64]
    r = subprocess.run(cmd + list(args), capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode:
        raise RuntimeError(f'git {args[0]} failed:\n{out}')
    return out

os.chdir('/kaggle/working')
if not os.path.exists('GenAI-A1'):
    git('clone', '-b', BRANCH, f'https://github.com/{REPO}.git')
os.chdir('GenAI-A1')
git('pull', 'origin', BRANCH)
print(git('log', '--oneline', '-3'))

In [ ]:
!pip install -q pytorch-msssim optuna mlflow onnx onnxruntime opencv-python-headless scikit-learn
!nvidia-smi | head -12

In [ ]:
# copy the Task 1-2 outputs (checkpoints + Optuna results) from the attached notebook output into ./outputs
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/t2_cls_v2/best.pt', recursive=True)
assert hits, 'attach the Tasks 1-2 notebook output as an input first'
src = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs('outputs', exist_ok=True)
for d in sorted(os.listdir(src)):
    if d.endswith('_v2') and not os.path.exists(f'outputs/{d}'):
        shutil.copytree(os.path.join(src, d), f'outputs/{d}')
print(sorted(os.listdir('outputs')))

In [ ]:
# data cache + a quick check that the Tasks 1-2 checkpoints load
!python -c "from src.data.pets import load_pets; d = load_pets('data'); print({k: v.shape for k, v in d.items()})"

In [ ]:
!GPU=0 TAG=_v2 TRIALS=20 TRIAL_EPOCHS=8 FINAL_EPOCHS=40 bash scripts/run_t3.sh

In [ ]:
# full test-set evaluation: input baseline, Task 1, Task 2 oracle / predicted routing, Task 3 soft MoE
!python -m src.eval.evaluate --t1 outputs/t1_v2/best.pt --cls outputs/t2_cls_v2/best.pt --specs outputs/spec_salt_v2/best.pt outputs/spec_blur_v2/best.pt outputs/spec_occlusion_v2/best.pt --moe outputs/t3_v2/best.pt --out outputs/eval_v2